# DeepCas9: 100 trials with the attached fixed split

This notebook mirrors the supplied split: seed 42, randomly select 10,699 seen samples, use all remaining samples as unseen, then split seen samples 80/20 into training and validation. All 100 trials use the same indices.

Required files in the notebook directory:
- `HTCas9_target_context_30nt.txt`: one 30-nt target-context sequence per line
- `HTCas9_indel_frequency_value_percentage.txt`: one activity value per line

In [ ]:
from pathlib import Path
import copy, json, random, warnings
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, Subset

SEQUENCE_FILE=Path("HTCas9_target_context_30nt.txt")
ACTIVITY_FILE=Path("HTCas9_indel_frequency_value_percentage.txt")
SPLIT_SEED=42
SELECTED_SEEN_SIZE=10699
VALIDATION_FRACTION=0.20
N_TRIALS=100
MODEL_SEED=42
MAX_EPOCHS=100
PATIENCE=10
OUTPUT_DIR=Path("results")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)

In [ ]:
def load_lines(path):
    if not path.exists(): raise FileNotFoundError(path.resolve())
    with open(path) as f: return [x.strip() for x in f if x.strip()]
sequences=np.asarray([x.upper() for x in load_lines(SEQUENCE_FILE)],dtype=object)
activities=np.asarray([float(x) for x in load_lines(ACTIVITY_FILE)],dtype=np.float32)
if len(sequences)!=len(activities): raise ValueError("Sequence/activity counts differ")
bad=[(i,s) for i,s in enumerate(sequences) if len(s)!=30 or not set(s)<=set("ACGT")]
if bad: raise ValueError(f"All inputs must be 30-nt A/C/G/T sequences. Examples: {bad[:10]}")
if len(sequences)<=SELECTED_SEEN_SIZE: raise ValueError(f"Need > {SELECTED_SEEN_SIZE} samples; found {len(sequences)}")
print("Samples:",len(sequences))

In [ ]:
np.random.seed(SPLIT_SEED)
full_indices=np.arange(len(activities))
selected_indices=np.random.choice(len(full_indices),size=SELECTED_SEEN_SIZE,replace=False)
unseen_indices=np.setdiff1d(full_indices,selected_indices)
train_indices,validation_indices=train_test_split(selected_indices,test_size=VALIDATION_FRACTION,random_state=SPLIT_SEED)
assert not(set(train_indices)&set(validation_indices))
assert not(set(train_indices)&set(unseen_indices))
assert not(set(validation_indices)&set(unseen_indices))
np.savetxt(OUTPUT_DIR/'train_indices.txt',train_indices,fmt='%d')
np.savetxt(OUTPUT_DIR/'validation_indices.txt',validation_indices,fmt='%d')
np.savetxt(OUTPUT_DIR/'unseen_indices.txt',unseen_indices,fmt='%d')
display(pd.DataFrame({'subset':['train','validation','unseen'],'n':[len(train_indices),len(validation_indices),len(unseen_indices)],'fraction':[len(train_indices)/len(full_indices),len(validation_indices)/len(full_indices),len(unseen_indices)/len(full_indices)]}))

In [ ]:
def export_subset(indices,name):
    d=OUTPUT_DIR/'split_data'/name; d.mkdir(parents=True,exist_ok=True)
    seq=sequences[indices]
    pd.Series(seq).to_csv(d/f'{name}_target_context_30nt.txt',index=False,header=False)
    pd.Series([s[4:24] for s in seq]).to_csv(d/f'{name}_target_20nt.txt',index=False,header=False)
    pd.Series([s[24:27] for s in seq]).to_csv(d/f'{name}_pam.txt',index=False,header=False)
    pd.Series(activities[indices]).to_csv(d/f'{name}_activity.txt',index=False,header=False)
    pd.DataFrame({'original_index':indices,'target_context_30nt':seq,'target_20nt':[s[4:24] for s in seq],'pam':[s[24:27] for s in seq],'activity':activities[indices]}).to_csv(d/f'{name}_complete.tsv',sep='\t',index=False)
for idx,name in [(train_indices,'train'),(validation_indices,'validation'),(unseen_indices,'unseen')]: export_subset(idx,name)

In [ ]:
B2I={'A':0,'C':1,'G':2,'T':3}
def encode(seqs):
    x=np.zeros((len(seqs),4,30),dtype=np.float32)
    for i,s in enumerate(seqs):
        for j,b in enumerate(s): x[i,B2I[b],j]=1
    return x
X=encode(sequences)
class SeqDataset(Dataset):
    def __init__(self,X,y): self.X=torch.tensor(X); self.y=torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i],self.y[i]
full_dataset=SeqDataset(X,activities)
train_set=Subset(full_dataset,train_indices); validation_set=Subset(full_dataset,validation_indices); unseen_set=Subset(full_dataset,unseen_indices)

In [ ]:
class DeepCas9(nn.Module):
    def __init__(self,f3=100,f5=70,f7=40,d1=80,d2=60,dropout=0.3):
        super().__init__()
        def branch(f,k): return nn.Sequential(nn.Conv1d(4,f,k),nn.ReLU(),nn.Dropout(dropout),nn.AvgPool1d(2,2,ceil_mode=True))
        self.b3,self.b5,self.b7=branch(f3,3),branch(f5,5),branch(f7,7)
        with torch.no_grad():
            z=torch.zeros(1,4,30); n=sum(b(z).flatten(1).shape[1] for b in [self.b3,self.b5,self.b7])
        self.fc1=nn.Linear(n,d1); self.fc2=nn.Linear(d1,d2); self.out=nn.Linear(d2,1); self.drop=nn.Dropout(dropout)
    def forward(self,x):
        z=torch.cat([self.b3(x).flatten(1),self.b5(x).flatten(1),self.b7(x).flatten(1)],1)
        z=self.drop(F.relu(self.fc1(z))); z=self.drop(F.relu(self.fc2(z)))
        return self.out(z).view(-1)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
def safe_corr(fn,y,p):
    if len(y)<2 or np.std(y)==0 or np.std(p)==0:return np.nan
    with warnings.catch_warnings(): warnings.simplefilter('ignore'); return float(fn(y,p)[0])
def predict(model,loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in loader:
            ps.extend(model(x.to(DEVICE)).cpu().numpy()); ys.extend(y.numpy())
    return np.asarray(ys),np.asarray(ps)

In [ ]:
trial_records=[]; trial_states={}
def objective(trial):
    set_seed(MODEL_SEED)
    params=dict(f3=trial.suggest_int('filters_k3',48,128,step=8),f5=trial.suggest_int('filters_k5',32,96,step=8),f7=trial.suggest_int('filters_k7',24,80,step=8),d1=trial.suggest_int('dense_1',64,160,step=16),d2=trial.suggest_int('dense_2',32,96,step=16),dropout=trial.suggest_float('dropout',0,0.5))
    lr=trial.suggest_float('learning_rate',1e-4,5e-3,log=True); bs=trial.suggest_categorical('batch_size',[16,32,64]); wd=trial.suggest_float('weight_decay',1e-8,1e-3,log=True)
    g=torch.Generator().manual_seed(MODEL_SEED)
    train_loader=DataLoader(train_set,batch_size=bs,shuffle=True,generator=g)
    val_loader=DataLoader(validation_set,batch_size=bs,shuffle=False)
    unseen_loader=DataLoader(unseen_set,batch_size=bs,shuffle=False)
    model=DeepCas9(**params).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=wd); loss_fn=nn.MSELoss()
    best=np.inf; state=None; best_epoch=0; counter=0
    for epoch in range(1,MAX_EPOCHS+1):
        model.train()
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True); loss=loss_fn(model(x),y); loss.backward(); opt.step()
        vy,vp=predict(model,val_loader); mse=float(mean_squared_error(vy,vp)); trial.report(mse,epoch)
        if mse<best: best=mse; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; best_epoch=epoch; counter=0
        else: counter+=1
        if counter>=PATIENCE: break
    model.load_state_dict(state); model.to(DEVICE); uy,up=predict(model,unseen_loader)
    pear=safe_corr(pearsonr,uy,up); spear=safe_corr(spearmanr,uy,up)
    trial.set_user_attr('best_epoch',best_epoch); trial_records.append({'trial':trial.number,'validation_mse':best,'unseen_pearson':pear,'unseen_spearman':spear}); trial_states[trial.number]=state
    print(f"Trial {trial.number:3d} | Validation MSE {best:.6f} | Unseen Pearson {pear:.4f} | Unseen Spearman {spear:.4f} | Epoch {best_epoch}")
    return best
study=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=42),study_name='DeepCas9_fixed_split_100_trials')
study.optimize(objective,n_trials=N_TRIALS)

In [ ]:
results=pd.DataFrame(trial_records).sort_values('trial').reset_index(drop=True)
results.to_csv(OUTPUT_DIR/'all_100_trial_metrics.csv',index=False)
results['validation_mse'].to_csv(OUTPUT_DIR/'Validation_loss.txt',index=False,header=False)
results['unseen_pearson'].to_csv(OUTPUT_DIR/'Unseen_Pearson.txt',index=False,header=False)
results['unseen_spearman'].to_csv(OUTPUT_DIR/'Unseen_Spearman.txt',index=False,header=False)
bn=study.best_trial.number; bp=study.best_trial.params; row=results.loc[results.trial==bn].iloc[0]
model=DeepCas9(f3=bp['filters_k3'],f5=bp['filters_k5'],f7=bp['filters_k7'],d1=bp['dense_1'],d2=bp['dense_2'],dropout=bp['dropout']); model.load_state_dict(trial_states[bn])
torch.save({'model_state_dict':model.state_dict(),'best_trial':bn,'best_params':bp,'best_epoch':study.best_trial.user_attrs['best_epoch'],'train_indices':train_indices,'validation_indices':validation_indices,'unseen_indices':unseen_indices,'validation_mse':float(row.validation_mse),'unseen_pearson':float(row.unseen_pearson),'unseen_spearman':float(row.unseen_spearman)},OUTPUT_DIR/'best_DeepCas9_model.pt')
with open(OUTPUT_DIR/'best_trial_summary.json','w') as f: json.dump({'best_trial':int(bn),'best_params':bp,'best_epoch':int(study.best_trial.user_attrs['best_epoch']),'validation_mse':float(row.validation_mse),'unseen_pearson':float(row.unseen_pearson),'unseen_spearman':float(row.unseen_spearman)},f,indent=2)
display(results.head()); print('Best trial:',bn); print(row); print('Saved to:',OUTPUT_DIR.resolve())